## 1. SparkSession - entry point

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (StructType, StructField, IntegerType, DoubleType, StringType)

import pandas as pd
import matplotlib.pyplot as plt

spark = (
    SparkSession.builder
        .appName("NinjaTraderETL")
        .master("local[*]")  # use all cores on machine as workers
        .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/26 09:12:48 WARN Utils: Your hostname, caroline.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.188 instead (on interface en0)
26/08/26 09:12:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/26 09:12:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Extract — read the raw CSV with an explicit schema (bronze layer)

In [4]:
raw_schema = StructType([
    StructField("trade_number", IntegerType(), True),
    StructField("qty", IntegerType(), True),
    StructField("entry_price", DoubleType(), True),
    StructField("exit_price", DoubleType(), True),
    StructField("entry_time_raw", StringType(), True),
    StructField("exit_time_raw", StringType(), True),
    StructField("entry_name", StringType(), True),
    StructField("exit_name", StringType(), True),
    StructField("profit_raw", StringType(), True),
    StructField("cum_net_profit_raw", StringType(), True),
    StructField("mae_raw", StringType(), True),
    StructField("mfe_raw", StringType(), True),
    StructField("bars", IntegerType(), True),
    StructField("_trailing", StringType(), True) # ninjatrader exports a trailing comma
])

bronze_df = (
    spark.read
    .option("header", True)
    .schema(raw_schema)
    .csv("trades.csv")
)

print(f"row count: {bronze_df.count()}")
bronze_df.printSchema()
bronze_df.show(5, truncate=False)

row count: 158
root
 |-- trade_number: integer (nullable = true)
 |-- qty: integer (nullable = true)
 |-- entry_price: double (nullable = true)
 |-- exit_price: double (nullable = true)
 |-- entry_time_raw: string (nullable = true)
 |-- exit_time_raw: string (nullable = true)
 |-- entry_name: string (nullable = true)
 |-- exit_name: string (nullable = true)
 |-- profit_raw: string (nullable = true)
 |-- cum_net_profit_raw: string (nullable = true)
 |-- mae_raw: string (nullable = true)
 |-- mfe_raw: string (nullable = true)
 |-- bars: integer (nullable = true)
 |-- _trailing: string (nullable = true)

+------------+---+-----------+----------+--------------------+--------------------+--------------+-------------+----------+------------------+-------+-------+----+---------+
|trade_number|qty|entry_price|exit_price|entry_time_raw      |exit_time_raw       |entry_name    |exit_name    |profit_raw|cum_net_profit_raw|mae_raw|mfe_raw|bars|_trailing|
+------------+---+-----------+----------+--

26/08/26 09:18:20 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Trade number, Qty, Entry price, Exit price, Entry time, Exit time, Entry name, Exit name, Profit, Cum. net profit, MAE, MFE, Bars, 
 Schema: trade_number, qty, entry_price, exit_price, entry_time_raw, exit_time_raw, entry_name, exit_name, profit_raw, cum_net_profit_raw, mae_raw, mfe_raw, bars, _trailing
Expected: trade_number but found: Trade number
CSV file: file:///Users/drew/code/data-eng/learn/smbc-spark/trades.csv


## 3. Transform — clean types, parse currency, derive columns (silver layer)

In [6]:
def parse_currency(colname: str):
    """convert ninjatrader-formatted currency stirngs like '$850.00' / '($850.00)' to signed double"""
    c = F.col(colname)
    is_negative = c.startswith("(")
    stripped = F.regexp_replace(c, r"[\$,()]","")
    numeric = stripped.cast("double")
    return F.when(is_negative, -numeric).otherwise(numeric)

TS_FORMAT = "M/d/yyyy h:mm:ss a"

silver_df = (
    bronze_df
    .withColumn("entry_time", F.to_timestamp("entry_time_raw", TS_FORMAT))
    .withColumn("exit_time", F.to_timestamp("exit_time_raw", TS_FORMAT))
    .withColumn("profit", parse_currency("profit_raw"))
    .withColumn("mae", parse_currency("mae_raw"))
    .withColumn("mfe", parse_currency("mfe_raw"))
    .withColumn(
        "side",
        F.when(F.col("entry_name").contains("long"), "long")
         .when(F.col("entry_name").contains("short"), "short")
         .otherwise("unknown")
    )
    .withColumn(
        "duration_minutes",
        (F.col("exit_time").cast("long") - F.col("entry_time").cast("long")) / 60.0
    )
    .withColumn("is_win", F.col("profit") > 0)
    .select(
        "trade_number", "qty", "side", "entry_price", "exit_price", "entry_time", "exit_time", "duration_minutes",
        "bars", "exit_name", "profit", "mae", "mfe", "is_win"
    )
)

silver_df.printSchema()
silver_df.orderBy("trade_number").show(10, truncate=False)

root
 |-- trade_number: integer (nullable = true)
 |-- qty: integer (nullable = true)
 |-- side: string (nullable = false)
 |-- entry_price: double (nullable = true)
 |-- exit_price: double (nullable = true)
 |-- entry_time: timestamp (nullable = true)
 |-- exit_time: timestamp (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- bars: integer (nullable = true)
 |-- exit_name: string (nullable = true)
 |-- profit: double (nullable = true)
 |-- mae: double (nullable = true)
 |-- mfe: double (nullable = true)
 |-- is_win: boolean (nullable = true)

+------------+---+-----+-----------+----------+-------------------+-------------------+------------------+----+-------------+------+-----+-----+------+
|trade_number|qty|side |entry_price|exit_price|entry_time         |exit_time          |duration_minutes  |bars|exit_name    |profit|mae  |mfe  |is_win|
+------------+---+-----+-----------+----------+-------------------+-------------------+------------------+----+-------------+

26/08/26 09:45:51 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Trade number, Qty, Entry price, Exit price, Entry time, Exit time, Entry name, Exit name, Profit, MAE, MFE, Bars
 Schema: trade_number, qty, entry_price, exit_price, entry_time_raw, exit_time_raw, entry_name, exit_name, profit_raw, mae_raw, mfe_raw, bars
Expected: trade_number but found: Trade number
CSV file: file:///Users/drew/code/data-eng/learn/smbc-spark/trades.csv


In [ ]:
trade_order = Window.orderBy("trade_number").rowsBetween(Window.unboundedPreceding, Window.currentRow)

gold_df = (
    silver_df
    .withColumn("cum_net_profit", F.round(F.sum("profit").over(trade_order), 2))
    .withColumn("running_max_equity", F.max("cum_net_profit").over(trade_order))
    .withColumn("drawdown",  F.round(F.col("cum_net_profit") - F.col("running_max_equity"), 2))
)

gold_df.select(
    "trade_number", "entry_time", "side", "profit", "cum_net_profit", "drawdown"
).orderBy("trade_number").show(10, truncate=False)

max_drawdown = gold_df.agg(F.min("drawdown")).first()[0]
print(f"max drawdown (spark-computed): ${max_drawdown:,.2f}")

{"ts": "2026-08-26 10:58:20.194", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `drawdown` cannot be resolved. Did you mean one of the following? [`bars`, `mae`, `side`, `is_win`, `mfe`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor29.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o273.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `drawdown` cannot be resolved. Did you mean one of the following? [`bars`, `mae`, `side`, `is_win`, `mfe`]. SQLSTATE: 42703;\n'Project [trade_number#14, entry_time#112, side#117, profit#114, cum_net_profit#175, 'drawdown]\n+- Project [trade_number#14, qty#15, side#117, entry_price#16,

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `drawdown` cannot be resolved. Did you mean one of the following? [`bars`, `mae`, `side`, `is_win`, `mfe`]. SQLSTATE: 42703;
'Project [trade_number#14, entry_time#112, side#117, profit#114, cum_net_profit#175, 'drawdown]
+- Project [trade_number#14, qty#15, side#117, entry_price#16, exit_price#17, entry_time#112, exit_time#113, duration_minutes#118, bars#26, exit_name#21, profit#114, mae#115, mfe#116, is_win#119, cum_net_profit#175, running_max_equity#178]
   +- Project [trade_number#14, qty#15, side#117, entry_price#16, exit_price#17, entry_time#112, exit_time#113, duration_minutes#118, bars#26, exit_name#21, profit#114, mae#115, mfe#116, is_win#119, cum_net_profit#175, running_max_equity#178, running_max_equity#178]
      +- Window [max(cum_net_profit#175) windowspecdefinition(trade_number#14 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) 